In [ ]:
cd ..

In [ ]:
import torch
import json
import os
from importlib.machinery import SourceFileLoader
from matplotlib import pyplot as plt
from utils.utils import *

In [ ]:
import skimage

def visualize(tensor, cmap='hsv', vmin=None, vmax=None, figsize=None, axis=True, title=None, psnr=None, ssim=None, l1=None):
    plt.figure(figsize=figsize)
    if (vmin is not None) and (vmax is not None):
        plt.imshow(tensor[0].permute(1,2,0).cpu().detach().numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
    else:
        plt.imshow(tensor[0].permute(1,2,0).cpu().detach().numpy(), cmap=cmap)
    if axis:
        plt.colorbar()
    else:
        plt.axis('off')
    title_string = title if title is not None else '' 
    title_string += f' l1: {l1:.3f}' if l1 is not None else ''
    title_string += f' psnr: {psnr:.3f}' if psnr is not None else ''
    title_string += f' ssim: {ssim:.3f}' if ssim is not None else ''
    plt.title(title_string)
    plt.show()

def grayscale(np_img):
    return (np_img[:,:,0]+np_img[:,:,1]+np_img[:,:,2])/3

def compute_psnr(GT, image):
    psnr = skimage.metrics.peak_signal_noise_ratio(
        GT[0].permute(1, 2, 0).cpu().detach().numpy(), image[0].permute(1, 2, 0).cpu().detach().numpy())
    return psnr

def compute_ssim(GT, image):
    ssim = skimage.metrics.structural_similarity(
        GT[0].mean(axis=0).cpu().detach().numpy(), image[0].mean(axis=0).cpu().detach().numpy())
    return ssim

In [ ]:
positional_encoding = True

from scipy.io import savemat, loadmat

ckpt_path = './de_occluding_broadband_metalens_neural_network_output'
args = json.load(open(ckpt_path + '/args.json'))
args = AttributeDict(args)

if positional_encoding:
    from models.localnet_positional_encoding import ParamLocal
else:
    from models.localnet import ParamLocal

concat=False

args.device = 'cuda:1'

saved_map_fname = 'training_state_050.pt'

net = ParamLocal(args).to(args.device)

new_state_dict = {}
saved_map_fname = 'training_state_050.pt' # broadband 
state_dict = torch.load(ckpt_path + '/' + saved_map_fname, map_location=args.device)['net_state_dict']
for k, v in state_dict.items():
    name = k.replace("module.", "") # 'module.layer1' -> 'layer1'
    new_state_dict[name] = v
net.load_state_dict(new_state_dict)
net.eval()

# Network inference

In [ ]:
import torchvision.transforms as transforms

transform_test = transforms.Compose([
    transforms.Resize([args.full_res, args.full_res]),
    transforms.ToTensor(),
])

from train_neural_network_ddp import PairedDataset, PairedRandomTransformWithPositionalEncoding
testset = PairedDataset(args.test_data_directory, transform=transform_test,
                        paired_transform=PairedRandomTransformWithPositionalEncoding((args.crop_patch_size, args.crop_patch_size),
                                                                                    (args.full_res, args.full_res), eval=True))
    
testloader = torch.utils.data.DataLoader(testset, batch_size=1, 
                                             shuffle=False,
                                             num_workers=4, 
                                             pin_memory=True,
                                             persistent_workers=True,
                                             prefetch_factor=4)

In [ ]:
import cv2
import tqdm
net = net.eval()

for step, batch_data in enumerate(tqdm.tqdm(testloader)):
    input, target = batch_data
    input = input.to(args.device)
    pred = net(input.to(torch.float32))
    if input.shape[1] > 3:
        input = input[:, :3, :, :]
    os.makedirs(os.path.join(ckpt_path, saved_map_fname[:-3]+'input'), exist_ok=True)
    os.makedirs(os.path.join(ckpt_path, saved_map_fname[:-3]+'inference_result'), exist_ok=True)
    cv2.imwrite(os.path.join(ckpt_path, saved_map_fname[:-3]+'input', f'img{step}.png'), cv2.cvtColor(np.clip(input.cpu().detach()[0].permute(1, 2, 0).numpy(), 0, 1)*255, cv2.COLOR_BGR2RGB))
    cv2.imwrite(os.path.join(ckpt_path, saved_map_fname[:-3]+'inference_result', f'img{step}.png'), cv2.cvtColor(np.clip(pred.cpu()[0].permute(1, 2, 0).detach().numpy(), 0, 1)*255, cv2.COLOR_BGR2RGB))